In [4]:
import torch
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [7]:
housing = fetch_california_housing()
X = housing['data']
y = housing['target']

In [9]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X,y)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full,y_train_full)

print(X_train.shape, X_test.shape, X_valid.shape)

scl = StandardScaler()
scl.fit(X_train)

X_train = scl.transform(X_train)
X_test = scl.transform(X_test)
X_valid = scl.transform(X_valid)

X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
X_valid = torch.FloatTensor(X_valid)

y_train = torch.FloatTensor(y_train).view(-1,1)
y_test = torch.FloatTensor(y_test).view(-1,1)
y_valid = torch.FloatTensor(y_valid).view(-1,1)

# y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
# y_test = torch.tensor(y_test, dtype=torch.float32).view(-1,1)
# y_valid = torch.tensor(y_valid, dtype=torch.float32).view(-1,1)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
valid_dataset = TensorDataset(X_valid, y_valid)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)
valid_loader = DataLoader(valid_dataset, batch_size=32)

(11610, 8) (5160, 8) (3870, 8)


In [10]:
import torch.nn as nn
import torchmetrics
import matplotlib.pyplot as plt
import numpy as np

In [11]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cpu'

In [33]:
def train(
	model, 
	optimizer, 
	criterion, 
	metric, 
	train_loader, 
	valid_loader, 
	n_epochs,
	clip_grad=False
	):
	history = {
		'loss' : [],
		'train_metric' : [],
		'valid_metric' : [],
	}
	for epoch in range(n_epochs):
		# Training 
		total_loss = 0
		metric.reset()
		for X_batch, y_batch in train_loader:
			X_batch, y_batch = X_batch.to(device), y_batch.to(device)
			model.train()
			y_pred = model(X_batch)
			loss = criterion(y_pred, y_batch)
			total_loss += loss.item()
			loss.backward()
			if clip_grad:
				nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
			optimizer.step()
			# for group in optimizer.param_groups:
			# 	for p in group['params']:
			# 		if p.grad is not None:
			# 			print(f'\t{p.grad.max()}') 
			optimizer.zero_grad()
			metric.update(y_pred, y_batch)
		
		avg_loss = total_loss / len(train_loader)
		history['loss'].append(avg_loss)

		avg_metric_train = metric.compute().item()
		history['train_metric'].append(avg_metric_train)

		# Evaluation 
		model.eval()
		metric.reset()
		with torch.no_grad():
			for X_batch, y_batch in valid_loader:
				X_batch, y_batch = X_batch.to(device), y_batch.to(device)
				y_pred = model(X_batch)
				metric.update(y_pred, y_batch)

		avg_metric_valid = metric.compute().item()
		history['valid_metric'].append(avg_metric_valid)

		print(
			f'Epoch: {epoch+1}/{n_epochs}, '
			+f'Loss: {round(avg_loss,3)}, '
			+f'Train Metric: {round(avg_metric_train,3)}, ' 
			+f'Valid Metric: {round(avg_metric_valid,3)}'
		)

		# if epoch>=2:
		# 	break
	return history

def plot_history(history, n_epochs, metric):
    plt.plot(np.arange(n_epochs) + 1, history['train_metric'], linestyle='--', color='r', marker='.', label='Train')
    plt.plot(np.arange(n_epochs) + 1, history['valid_metric'], linestyle='--', color='b', marker='.', label='Valid')
    plt.legend()
    plt.grid()
    plt.xlabel('Epochs')
    plt.ylabel(f'{metric.__class__.__name__}')
    plt.show()

In [34]:
learning_rate = 0.085
n_epochs = 20
model = nn.Sequential(
	nn.Linear(in_features=8, out_features=50), 
	nn.LeakyReLU(),
	nn.Linear(in_features=50, out_features=100), 
	nn.LeakyReLU(),
	nn.Linear(in_features=100, out_features=100), 
	nn.LeakyReLU(),
	nn.Linear(in_features=100, out_features=50), 
	nn.LeakyReLU(),
	nn.Linear(in_features=50, out_features=1),
).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=learning_rate)
metric = torchmetrics.MeanAbsoluteError().to(device)

history = train(
    model, 
	optimizer, 
	criterion, 
	metric, 
	train_loader, 
	valid_loader, 
	n_epochs, 
	clip_grad = True,
    )

Epoch: 1/20, Loss: 0.707, Train Metric: 0.59, Valid Metric: 0.495
Epoch: 2/20, Loss: 0.443, Train Metric: 0.475, Valid Metric: 0.424
Epoch: 3/20, Loss: 0.408, Train Metric: 0.454, Valid Metric: 0.432
Epoch: 4/20, Loss: 0.385, Train Metric: 0.44, Valid Metric: 0.419
Epoch: 5/20, Loss: 0.373, Train Metric: 0.433, Valid Metric: 0.419
Epoch: 6/20, Loss: 0.36, Train Metric: 0.422, Valid Metric: 0.478
Epoch: 7/20, Loss: 0.358, Train Metric: 0.42, Valid Metric: 0.39
Epoch: 8/20, Loss: 0.35, Train Metric: 0.416, Valid Metric: 0.379
Epoch: 9/20, Loss: 0.339, Train Metric: 0.407, Valid Metric: 0.378
Epoch: 10/20, Loss: 0.331, Train Metric: 0.402, Valid Metric: 0.381
Epoch: 11/20, Loss: 0.326, Train Metric: 0.399, Valid Metric: 0.383
Epoch: 12/20, Loss: 0.323, Train Metric: 0.395, Valid Metric: 0.382
Epoch: 13/20, Loss: 0.321, Train Metric: 0.393, Valid Metric: 0.382
Epoch: 14/20, Loss: 0.316, Train Metric: 0.389, Valid Metric: 0.407
Epoch: 15/20, Loss: 0.314, Train Metric: 0.389, Valid Metric: 0

tensor([[1.2270],
        [2.8260],
        [1.7790],
        [0.6600],
        [2.0630],
        [1.6670],
        [0.9600],
        [3.5910],
        [0.8280],
        [0.6190],
        [1.9060],
        [3.3790],
        [2.8180],
        [1.0930],
        [0.7340],
        [3.5310],
        [2.2110],
        [5.0000],
        [1.7070],
        [1.1060],
        [2.7610],
        [1.6810],
        [1.5180],
        [2.2580],
        [1.0910],
        [0.7730],
        [1.9130],
        [2.2050],
        [2.4480],
        [1.5890],
        [2.7660],
        [2.0120]])